#Merge per-donor h5ads into all-samples h5ads

In [ ]:
print("Hello World")

In [1]:
import logging
logging.getLogger("fontTools").setLevel(logging.WARNING)

import os
_r = os.path.abspath(".")
while _r != os.path.dirname(_r) and not os.path.exists(os.path.join(_r, ".notebooks_root")):
    _r = os.path.dirname(_r)
os.chdir(_r if os.path.exists(os.path.join(_r, ".notebooks_root")) else "/oak/stanford/groups/quake/mmantri/group.quake/tabula_longread/notebooks")  # cd to notebooks/ root (marked .notebooks_root); relocation-proof

import os, glob, gc
import scanpy as sc
import anndata as ad
import pandas as pd
gc.enable()

In [4]:
H5AD_DIR = "./../pacbio/h5ads"

# (flavor, kind) pairs match the filenames produced by 01_01/00b.
# kind="genes"   -> from 01_01 notebooks
# kind in (pbids, ensemblids, ensemblids_annotatedonly) -> from 01_03 notebooks
CONFIGS = [
    ("recollapsed",   "genes"),
    # ("recollapsed",   "pbids"),
    # ("recollapsed",   "ensemblids"),
    # ("recollapsed",   "ensemblids_annotatedonly"),
    # ("unsyncronized", "genes"),
    # ("unsyncronized", "pbids"),
    # ("unsyncronized", "ensemblids"),
    # ("unsyncronized", "ensemblids_annotatedonly"),
]

for flavor, kind in CONFIGS:
    suffix = f"_pacbio_{flavor}_raw_counts_{kind}_bc_anndata_mincounts500.h5ad"
    # Per-donor files are prefixed with the donor id (e.g. TSP33_pacbio_recollapsed_...).
    # Skip the merged all_samples_* file if it already exists in the directory.
    paths = sorted(
        p for p in glob.glob(f"{H5AD_DIR}/*{suffix}")
        if not os.path.basename(p).startswith("all_samples_")
    )
    if not paths:
        print(f"\n[skip] {flavor}/{kind}: no per-donor files match *{suffix}")
        continue

    print(f"\n=== merging {flavor}/{kind} ({len(paths)} donors) ===")
    adatas = []
    keys = []
    for p in paths:
        donor = os.path.basename(p).split("_")[0]
        a = sc.read_h5ad(p)
        print(f"  {donor}: {a.shape}")
        adatas.append(a)
        keys.append(donor)

    # Per-donor obs names already include the tissue suffix (e.g. barcode_T5);
    # tissues are partitioned across donors so no further uniquification is needed.
    #
    # NOTE: ad.concat(..., merge="first") copies var annotation from ONLY the first object, so any
    # feature that appears only in a later donor comes out with NaN gene_name/transcript_id (even
    # though that donor has it). Concat with merge=None (drop var annot) and rebuild .var from ALL
    # donors: stack every per-donor .var, keep the first row per feature (per-donor annots are
    # complete), and align to the merged var order.
    merged = ad.concat(adatas, join="outer", merge=None)
    all_var = pd.concat([a.var for a in adatas], join="outer")
    all_var = all_var[~all_var.index.duplicated(keep="first")]
    merged.var = all_var.reindex(merged.var_names)
    n_missing = int(merged.var["gene_name"].isna().sum()) if "gene_name" in merged.var else -1
    print(f"  merged: {merged.shape}  (gene_name still missing: {n_missing})")

    out_path = f"{H5AD_DIR}/all_samples{suffix}"
    merged.write_h5ad(out_path)
    print(f"  saved: {out_path}")

    del adatas, merged, all_var
    gc.collect()


=== merging recollapsed/genes (12 donors) ===
  TSP10: (4057, 22176)
  TSP19: (7197, 26211)
  TSP21: (28570, 41371)
  TSP25: (31015, 54253)
  TSP27: (80409, 46498)
  TSP28: (21652, 31973)
  TSP2: (11750, 31360)
  TSP32: (3222, 20722)
  TSP33: (85506, 61479)
  TSP4: (5373, 30150)
  TSP7: (9881, 23733)
  TSP9: (4869, 26047)
  merged: (293501, 69580)  (gene_name still missing: 0)
  saved: ./../pacbio/h5ads/all_samples_pacbio_recollapsed_raw_counts_genes_bc_anndata_mincounts500.h5ad


  TSP10: (4057, 22176)


  TSP19: (7197, 26211)


  TSP21: (28570, 41371)


  TSP25: (31015, 54253)


...

  TSP4: (5371, 100017)


  TSP7: (9885, 80777)


  TSP9: (4861, 87714)


  merged: (293433, 202845)  (gene_name still missing: 0)


  saved: ./../pacbio/h5ads/all_samples_pacbio_unsyncronized_raw_counts_ensemblids_annotatedonly_bc_anndata_mincounts500.h5ad


In [5]:
merged1 = sc.read_h5ad("./../pacbio/h5ads/all_samples_pacbio_recollapsed_raw_counts_genes_bc_anndata_mincounts500.h5ad")
print(merged1.shape)

(293501, 69580)


In [6]:
# ---------------------------------------------------------------------------
# Helpers to save pigeon and SQANTI3 classification CSVs at three levels that
# mirror the three h5ad files:
#   - pbids: one row per PB.X.Y     (dedup by `isoform`)
#   - ensemblids: one row per transcript_id (ENST if available, else PB.X.Y)
#   - ensemblids_annotatedonly: subset of ensemblids where transcript_id startswith "ENST"
#
# The pbid -> transcript_id map is rebuilt from the per-tissue pigeon
# annotated.info.csv files (same logic as the h5ad-building loop above).
#
# For ensemblids/ensemblids_annotatedonly CSVs, the `isoform` column is set to transcript_id
# (instead of the original PB.X.Y) so that add_classification() can join on it
# when loading into the ensemblids or ensemblids_annotatedonly h5ads (whose var_names are
# transcript_ids). The original PB.X.Y is preserved as `molecule_id`.
# ---------------------------------------------------------------------------

# Tube IDs are sourced from sample_metadata.csv (not folder listings) so that
# the iteration order is reproducible and tissues with missing outputs are
# silently skipped.
_SAMPLE_METADATA_TUBE_IDS = sorted(
    pd.read_csv("./../csvs/sample_metadata.csv")["tube_id"].dropna().unique()
)

def _build_transcript_map(base_dir):
    """Return dict: pbid -> transcript_id (ENST if available, else pbid)."""
    seurat_dir = os.path.join(base_dir, "seurat")
    frames = []
    for tissue in _SAMPLE_METADATA_TUBE_IDS:
        path = os.path.join(seurat_dir, tissue, f"{tissue}.annotated.info.csv")
        if not os.path.exists(path):
            print(f"  transcript map: skipping {tissue} (no annotated.info.csv)")
            continue
        frames.append(pd.read_csv(path, sep="\t", usecols=["pbid", "transcript"]))
    annot = pd.concat(frames, ignore_index=True).drop_duplicates(subset="pbid")
    annot["transcript_id"] = annot["transcript"].where(
        annot["transcript"].astype(str).str.startswith("ENST"),
        annot["pbid"],
    )
    return dict(zip(annot["pbid"], annot["transcript_id"]))


def _concat_per_tissue_cls(base_dir, subdir, filename_fn, drop_cols=()):
    """Read per-tissue classification files, tag with `tissue_source`, concat."""
    root = os.path.join(base_dir, subdir)
    frames = []
    for tissue in _SAMPLE_METADATA_TUBE_IDS:
        path = os.path.join(root, tissue, filename_fn(tissue))
        if not os.path.exists(path):
            print(f"  {subdir}: skipping {tissue} (file not found)")
            continue
        df = pd.read_csv(path, sep="\t", dtype=str)
        df["tissue_source"] = tissue
        frames.append(df)
        print(f"  {tissue}: {len(df):,} rows")
    cls = pd.concat(frames, ignore_index=True)
    for col in drop_cols:
        if col in cls.columns:
            cls = cls.drop(columns=[col])
    return cls


def _save_three_levels(cls_df, transcript_map, out_prefix):
    """Write pbids / ensemblids / ensemblids_annotatedonly CSVs from a concatenated cls_df.

    - molecules: dedup by `isoform` column (the PB.X.Y ID from pigeon/SQANTI3).
      Saved to `{out_prefix}.csv` (no suffix) for backward compatibility with
      existing downstream notebooks that load `..._classification.csv`.
    - transcripts: add `transcript_id` via transcript_map, dedup by
      `transcript_id`, then set `isoform` = `transcript_id` (so that
      add_classification() can join directly against the ensemblids h5ad
      whose var_names are transcript_ids). The original PB.X.Y is kept as
      `molecule_id`. Saved to `{out_prefix}_transcripts.csv`.
    - isoforms: subset of the transcripts table where transcript_id starts
      with "ENST" (i.e. only reference-matched transcripts). Saved to
      `{out_prefix}_isoforms.csv`.
    """
    # pbids level
    mol = cls_df.drop_duplicates(subset="isoform")
    mol_path = f"{out_prefix}.csv"
    mol.to_csv(mol_path, index=False)
    print(f"  pbids: {len(mol):>10,} rows -> {mol_path}")

    # ensemblids level (map pbid -> transcript_id, then dedup)
    tr = cls_df.copy()
    tr["transcript_id"] = (
        tr["isoform"].map(transcript_map).fillna(tr["isoform"])
    )
    tr = tr.drop_duplicates(subset="transcript_id")
    # Set isoform = transcript_id so add_classification() can join on it;
    # keep the original PB.X.Y as molecule_id for reference
    tr = tr.rename(columns={"isoform": "molecule_id"})
    tr["isoform"] = tr["transcript_id"]
    tr_path = f"{out_prefix}_transcripts.csv"
    tr.to_csv(tr_path, index=False)
    print(f"  ensemblids: {len(tr):>10,} rows -> {tr_path}")

    # ensemblids_annotatedonly level (ENST only)
    iso = tr[tr["transcript_id"].astype(str).str.startswith("ENST")]
    iso_path = f"{out_prefix}_isoforms.csv"
    iso.to_csv(iso_path, index=False)
    print(f"  ensemblids_annotatedonly: {len(iso):>10,} rows -> {iso_path}")

In [ ]:
# Accumulate pigeon and SQANTI3 classification CSVs at all three aggregation
# levels (pbids / ensemblids / ensemblids_annotatedonly) for both the full-cohort and
# TSP33-only recollapsed datasets.
#
# Output filenames (in ./../csvs/):
#   {label}_recollapsed_{tool}_classification.csv              (molecules)
#   {label}_recollapsed_{tool}_classification_ensemblids.csv  (transcripts)
#   {label}_recollapsed_{tool}_classification_ensemblids_annotatedonly.csv     (isoforms, ENST-only)
# where label in {"all_samples", "TSP33"} and tool in {"pigeon", "sqanti3"}.
#
# The pbids-level SQANTI3 files overwrite the existing
# `{label}_recollapsed_sqanti3_classification.csv` produced in the cells above;
# downstream notebooks that load that path via `add_classification()` are
# unaffected.
#
# NOTE: SQANTI3 is read from the RulesFilter output
# (`{t}_RulesFilter_classification.txt`) rather than the QC output
# (`{t}_classification.txt`). Same rows/isoforms (verified identical ID set and
# count), but it adds the `filter_result` column (Isoform vs Artifact) so the
# merged CSVs now carry the artifact call. (RulesFilter is a pandas re-write:
# LF line endings, booleans as True/False; pandas already normalizes NA/empty.)

os.makedirs("./../csvs", exist_ok=True)

DATASETS = [
    ("./../pacbio/recollapsed/", "all_samples"),
]

# Columns to drop from SQANTI3 classification (large sequence fields)
SQANTI3_DROP = ("ORF_seq", "seq_A_downstream_TTS")

for base_dir, label in DATASETS:
    print(f"\n=== {label}  ({base_dir}) ===")

    print("Building pbid -> transcript_id map...")
    transcript_map = _build_transcript_map(base_dir)
    print(f"  transcript map: {len(transcript_map):,} PB IDs")

    # ---- pigeon ----
    print("\n[pigeon] reading per-tissue classification files...")
    pigeon_cls = _concat_per_tissue_cls(
        base_dir,
        subdir="classify",
        filename_fn=lambda t: "collapse_classification.filtered_lite_classification.txt",
    )
    print(f"  total rows before dedup: {len(pigeon_cls):,}")
    _save_three_levels(
        pigeon_cls,
        transcript_map,
        out_prefix=f"./../csvs/{label}_recollapsed_pigeon_classification",
    )
    del pigeon_cls
    gc.collect()

    # ---- SQANTI3 ----
    print("\n[sqanti3] reading per-tissue classification files...")
    sqanti3_cls = _concat_per_tissue_cls(
        base_dir,
        subdir="sqanti3",
        filename_fn=lambda t: f"{t}_RulesFilter_classification.txt",
        drop_cols=SQANTI3_DROP,
    )
    print(f"  total rows before dedup: {len(sqanti3_cls):,}")
    _save_three_levels(
        sqanti3_cls,
        transcript_map,
        out_prefix=f"./../csvs/{label}_recollapsed_sqanti3_classification",
    )
    del sqanti3_cls
    gc.collect()

print("\nDone.")


=== all_samples  (./../pacbio/recollapsed/) ===
Building pbid -> transcript_id map...
  transcript map: skipping T26 (no annotated.info.csv)
  transcript map: skipping T27 (no annotated.info.csv)
  transcript map: 11,529,975 PB IDs

[pigeon] reading per-tissue classification files...
  T10: 878,235 rows
  T12: 819,554 rows
  T13: 172,908 rows
  T14: 471,216 rows
  T15: 642,645 rows
  T16: 2,343,300 rows
  T17: 760,146 rows
  T19: 589,779 rows
  T20: 104,613 rows
  T21: 69,880 rows
  T22: 65,275 rows
  T23: 616,126 rows
  T25: 735,308 rows
  classify: skipping T26 (file not found)
  classify: skipping T27 (file not found)
  T28: 295,134 rows
  T29: 676,312 rows
  T3: 791,661 rows
  T30: 763,240 rows
  T31: 776,527 rows
  T32: 838,157 rows
  T33: 415,460 rows
  T34: 653,430 rows
  T35: 627,184 rows
  T37: 269,659 rows
  T38: 487,763 rows
  T39: 959,016 rows
  T4: 461,858 rows
  T40: 602,827 rows
  T41: 545,100 rows
  T42: 679,364 rows
  T43: 206,918 rows
  T44: 873,591 rows
  T45: 2,189

IOStream.flush timed out


  transcripts: 10,200,660 rows -> ./../csvs/all_samples_recollapsed_pigeon_classification_ensemblids.csv
  isoforms:       201,109 rows -> ./../csvs/all_samples_recollapsed_pigeon_classification_ensemblids_annotatedonly.csv

[sqanti3] reading per-tissue classification files...
  T10: 857,600 rows
  T12: 789,459 rows
  T13: 167,598 rows
  T14: 452,475 rows
  T15: 624,157 rows
  T16: 2,326,679 rows
  T17: 737,520 rows
  T19: 570,141 rows
  T20: 101,740 rows
  T21: 67,523 rows
  T22: 62,872 rows
  T23: 596,548 rows
  T25: 708,539 rows
  sqanti3: skipping T26 (file not found)
  sqanti3: skipping T27 (file not found)
  T28: 288,839 rows
  T29: 655,840 rows
